# 02 — Fitness-Level Classification**Goal:** train a 3-class classifier that places a user into Beginner / Intermediate / Advanced based on their physical profile and self-reported activity. The classifier output drives `/predict/fitness-level` and the recommender's level filter on `/recommend`.**Output artifact:** `ai_models/ml_models/fitness_classifier.pkl` (+ `fitness_classifier_labels.pkl`).**Algorithm:** Gradient Boosting Classifier with StandardScaler. Tree ensembles handle the non-linear cutoffs (e.g. "high activity but high BMI → still intermediate") that a linear model misses.**Inputs (5):** `age, weight_kg, height_cm, activity_level, gender`**Labels (3):** `0 = Beginner, 1 = Intermediate, 2 = Advanced` — same encoding the API returns.

In [ ]:
import sys, pathlib, json, timeROOT = pathlib.Path('..').resolve()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport joblibfrom sklearn.ensemble import GradientBoostingClassifierfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFoldfrom sklearn.metrics import (accuracy_score, f1_score, classification_report,                             confusion_matrix, ConfusionMatrixDisplay)sns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (9, 5)print('sklearn imports OK')

## 1. Load the data

In [ ]:
df = pd.read_csv(ROOT / 'datasets' / 'fitness_profiles.csv')print(f'Loaded {len(df)} rows')# Class balancecounts = df['fitness_level'].value_counts().sort_index()print('Class balance:'); print(counts)plt.figure(figsize=(5, 3))sns.barplot(x=counts.index, y=counts.values, palette=['#00ff88','#00d4ff','#7b5cff'])plt.xticks([0,1,2], ['Beginner', 'Intermediate', 'Advanced'])plt.title('Class distribution'); plt.ylabel('count'); plt.tight_layout(); plt.show()

## 2. Feature inspection

In [ ]:
FEATURES = ['age','weight_kg','height_cm','activity_level','gender']# Pairwise look at how features separate the classesfig, axes = plt.subplots(1, 3, figsize=(14, 4))for ax, col in zip(axes, ['age','activity_level','weight_kg']):    for lvl, color, lbl in [(0,'#00ff88','Beginner'),(1,'#00d4ff','Intermediate'),(2,'#7b5cff','Advanced')]:        ax.hist(df.loc[df['fitness_level']==lvl, col], bins=20, alpha=0.5, label=lbl, color=color)    ax.set_title(col); ax.legend(fontsize=8)plt.tight_layout(); plt.show()

## 3. Train / test split (stratified)

In [ ]:
X = df[FEATURES]y = df['fitness_level'].astype(int)X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f'Train: {len(X_train)} · Test: {len(X_test)}')print('Train class %:', (y_train.value_counts(normalize=True).sort_index() * 100).round(1).to_dict())

## 4. Train Gradient Boosting with grid-search CV

In [ ]:
pipe = Pipeline([    ('scaler', StandardScaler()),    ('clf',    GradientBoostingClassifier(random_state=42)),])param_grid = {    'clf__n_estimators': [100, 200],    'clf__max_depth':    [3, 5],    'clf__learning_rate': [0.05, 0.1],}grid = GridSearchCV(    pipe, param_grid,    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),    scoring='f1_macro', n_jobs=-1,)t0 = time.time()grid.fit(X_train, y_train)print(f'Trained in {time.time()-t0:.1f}s')print(f'Best params: {grid.best_params_}')print(f'CV F1 macro: {grid.best_score_:.4f}')best = grid.best_estimator_

## 5. Test-set evaluation

In [ ]:
y_pred = best.predict(X_test)acc = accuracy_score(y_test, y_pred)f1  = f1_score(y_test, y_pred, average='macro')print(f'Accuracy   : {acc:.4f}')print(f'F1 (macro) : {f1:.4f}')print()print(classification_report(    y_test, y_pred,    target_names=['Beginner','Intermediate','Advanced']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)fig, ax = plt.subplots(figsize=(5.5, 5))ConfusionMatrixDisplay(cm, display_labels=['Beg','Inter','Adv']).plot(ax=ax, cmap='viridis')plt.title('Confusion matrix (test set)'); plt.tight_layout(); plt.show()

## 6. Feature importance from the GBM

In [ ]:
clf = best.named_steps['clf']imp = pd.DataFrame({'feature': FEATURES, 'importance': clf.feature_importances_})imp = imp.sort_values('importance', ascending=False)print(imp.to_string(index=False))plt.figure(figsize=(7, 3.5))sns.barplot(data=imp, x='importance', y='feature', palette='magma')plt.title('GBM feature importance'); plt.tight_layout(); plt.show()

## 7. Probability calibration sanity-check

In [ ]:
# A spot-check: how confident is the model in correct predictions vs wrong ones?proba = best.predict_proba(X_test)top_p = proba.max(axis=1)correct = (y_pred == y_test)plt.figure(figsize=(8, 3.5))plt.hist(top_p[correct],   bins=20, alpha=0.6, label='correct',   color='#00ff88')plt.hist(top_p[~correct],  bins=20, alpha=0.6, label='incorrect', color='#ff6b35')plt.xlabel('top-class probability'); plt.ylabel('count')plt.title('Confidence on correct vs incorrect test samples')plt.legend(); plt.tight_layout(); plt.show()print(f'Mean confidence on correct preds:   {top_p[correct].mean():.3f}')print(f'Mean confidence on incorrect preds: {top_p[~correct].mean():.3f}')

## 8. Save the model + label map

In [ ]:
out_dir = ROOT / 'ai_models' / 'ml_models'out_dir.mkdir(parents=True, exist_ok=True)joblib.dump(best, out_dir / 'fitness_classifier.pkl')joblib.dump({0:'Beginner', 1:'Intermediate', 2:'Advanced'},            out_dir / 'fitness_classifier_labels.pkl')report = {    'model': 'fitness_classifier',    'algorithm': 'GradientBoosting + StandardScaler',    'best_params': grid.best_params_,    'cv_f1_macro': float(grid.best_score_),    'test_accuracy': float(acc),    'test_f1_macro': float(f1),    'features': FEATURES,    'classes': ['Beginner', 'Intermediate', 'Advanced'],    'trained_at': time.strftime('%Y-%m-%dT%H:%M:%S'),}(out_dir / 'reports').mkdir(exist_ok=True)(out_dir / 'reports' / f'fitness_classifier_{time.strftime("%Y%m%d_%H%M%S")}.json').write_text(json.dumps(report, indent=2))print('saved fitness_classifier.pkl + label map + report')

## 9. Smoke-test through the running service

In [ ]:
import backend.services.ml_service as ml_modml_mod.get_ml_service.cache_clear()from backend.services.ml_service import get_ml_servicesvc = get_ml_service()cases = [    ('Inactive 50yo male, BMI 30',  dict(age=50, weight_kg=92, height_cm=175, activity_level=1, gender=1)),    ('Active 25yo female, BMI 22',   dict(age=25, weight_kg=60, height_cm=165, activity_level=4, gender=0)),    ('Athlete 30yo male',           dict(age=30, weight_kg=80, height_cm=183, activity_level=5, gender=1)),]for name, kw in cases:    res = svc.classify_fitness(**kw)    print(f'{name:35s} → {res["level_name"]:13s}  conf={res["confidence"]:.2f}')

---The `/predict/fitness-level` endpoint will now serve predictions from this model. The recommender's level-based exercise selection on `/recommend` is keyed on the `level_id` we just learned to predict.